# Proje Akışı

- 1. Veri çekme + metin EDA

- 2. Ön işleme (lowercase, regex, tokenize, stopword)

- 3. BoW vs TF-IDF karşılaştırma

- 4. Sınıflandırma (TF-IDF + LogReg -> 5 kategori)

- 5. NER (spaCy)  -- haberdeki varlıklar

- 6. Semantik benzerlik (sentence-transformers)

- 7. Modeli kaydet + modern transformer notu

In [8]:
# 1) VERİ ÇEKME (cache'li) -- gerçek dünyada veri URL/API'den gelir
import os, urllib.request
import pandas as pd

def fetch_bbc(cache_dir = 'data'):
    os.makedirs(cache_dir, exist_ok= True)
    path = os.path.join(cache_dir, "bbc-news.csv")
    if not os.path.exists(path):
        url = ("https://raw.githubusercontent.com/suraj-deshmukh/BBC-Dataset-News-Classification/master/dataset/dataset.csv")
        urllib.request.urlretrieve(url, path)
    return path

# BBC İngiliz kaynağı kaynağı -> latin-1 encoding
df = pd.read_csv(fetch_bbc(), encoding= 'latin-1')

df.columns = ['text','category'] # 'news', 'type' -< anlaşılır isimler
print("Boyut:", df.shape)
print(df["category"].value_counts())

Boyut: (2225, 2)
category
sport            511
business         510
politics         417
tech             401
entertainment    386
Name: count, dtype: int64


In [9]:
# 2) METİN EDA - dokğüman uzunluğu + en sık kelimeler
df["n_words"] = df["text"].str.split().str.len()
print("Ortalama döküman uzunluğu:", int(df['n_words'].mean()), "kelime")
print("En kısa / en uzun:", df["n_words"].min(), "/", df["n_words"].max())

# En sık kelimeler (stopword'siz) - CountVectorizer ile hızlı bakış
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(stop_words= 'english', max_features= 10)
cv.fit(df['text'])
print("\nEn sık 10 kelime (stopword'süz):", list(cv.get_feature_names_out()))

Ortalama döküman uzunluğu: 384 kelime
En kısa / en uzun: 89 / 4432

En sık 10 kelime (stopword'süz): ['government', 'mr', 'new', 'people', 'said', 'time', 'uk', 'world', 'year', 'years']


In [10]:
# ÖN İŞLEME - bir haberde adım adım göster (NLP'nin kalbi)

import re
import nltk
nltk.download('stopwords', quiet= True)
from nltk.corpus import stopwords
stops = set(stopwords.words("english"))

ornek = df["text"].iloc[0][:200]
print("HAM\n", ornek, "\n")

metin = ornek.lower()               # 1. kücuk harf
metin = re.sub(r'[^a-z\s]', '', metin) # 2. harf dışını temizle (sayı/noktalama)
tokens = metin.split()          # 3. tokenize (kelimelere böl)
temiz = [t for t in tokens if t not in stops] # 4. stopword çıkar

print("TEMİZ TOKEN'LAR:\n", temiz[:25])

HAM
 China had role in Yukos split-up
 
 China lent Russia $6bn (Â£3.2bn) to help the Russian government renationalise the key Yuganskneftegas unit of oil group Yukos, it has been revealed.
 
 The Kremlin  

TEMİZ TOKEN'LAR:
 ['china', 'role', 'yukos', 'splitup', 'china', 'lent', 'russia', 'bn', 'bn', 'help', 'russian', 'government', 'renationalise', 'key', 'yuganskneftegas', 'unit', 'oil', 'group', 'yukos', 'revealed', 'kremlin']


In [11]:
# 4) BoW vs TF-IDF -- aynı sınıflandırıcı, iki temsil
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

X_train,X_test,y_train,y_test = train_test_split(
    df['text'], df["category"], test_size= 0.2, stratify= df["category"], random_state= 42
)

pipelines = [
    ("BoW (CountVectorizer)", CountVectorizer(stop_words= 'english')),
    ("TF-IDF", TfidfVectorizer(stop_words= 'english'))
]

for ad, vec in pipelines:
    pipe = Pipeline([('vec', vec), ('clf', LogisticRegression(max_iter= 1000))])
    pipe.fit(X_train,y_train)
    acc = accuracy_score(y_test, pipe.predict(X_test))
    print(f"{ad:<24} accuracy: {acc:.3f}")

BoW (CountVectorizer)    accuracy: 0.969
TF-IDF                   accuracy: 0.980


In [13]:
# 5) SINIFLANDIRMA -- TF-IDF + LogReg (5 kategori) ve detaylı rapor
from sklearn.metrics import classification_report, confusion_matrix

model = Pipeline([('vec', TfidfVectorizer(stop_words= 'english', ngram_range= (1,2),
                                           min_df= 3, sublinear_tf= True)),
                                          ('clf', LogisticRegression(max_iter= 1000, C = 10))])

model.fit(X_train,y_train)
y_pred = model.predict(X_test)

print(classification_report(y_test,y_pred))
print("Confusion matrix (satır = gercek, sütun = tahmin):")
print(pd.DataFrame(confusion_matrix(y_test,y_pred),
                   index= model.classes_, columns= model.classes_))

               precision    recall  f1-score   support

     business       0.98      0.96      0.97       102
entertainment       0.97      1.00      0.99        77
     politics       0.95      1.00      0.98        84
        sport       0.99      0.99      0.99       102
         tech       1.00      0.95      0.97        80

     accuracy                           0.98       445
    macro avg       0.98      0.98      0.98       445
 weighted avg       0.98      0.98      0.98       445

Confusion matrix (satır = gercek, sütun = tahmin):
               business  entertainment  politics  sport  tech
business             98              1         3      0     0
entertainment         0             77         0      0     0
politics              0              0        84      0     0
sport                 1              0         0    101     0
tech                  1              1         1      1    76


In [15]:
# 6) NER -- spaCy ile bir haberde geçen varlıkları çıkar
import spacy
nlp = spacy.load("en_core_web_sm")

# Bir business haberi seç (şirket/kişi/yer dolu olur)
biz = df[df["category"] == 'business']['text'].iloc[0]
doc = nlp(biz[:1000]) # ilk 100 karakter (hız)


print("Haberde geçen varlıklar (tip -> metinler):")
from collections import defaultdict
gruplar = defaultdict(list)
for ent in doc.ents:
    gruplar[ent.label_].append(ent.text)
for tip in ["ORG","PERSON","AGE","MONEY","DATE"]:
    if gruplar[tip]:
        print(f"    {tip:<7} -> {list(dict.fromkeys(gruplar[tip]))[:3]}")


Haberde geçen varlıklar (tip -> metinler):
    ORG     -> ['Yuganskneftegas', 'Kremlin', 'VEB']
    PERSON  -> ['Yukos', 'CNPC']
    MONEY   -> ['6bn']
    DATE    -> ['Tuesday']


In [18]:
# 7) SEMANTİK BENZERLİK -- "bu habere en benzer haber" (mini arama motoru)

from sentence_transformers import SentenceTransformer, util

st = SentenceTransformer('all-MiniLM-L6-v2')

ornek_df = df.sample(400, random_state= 42).reset_index(drop= True)
embeddings = st.encode(ornek_df['text'].tolist(), show_progress_bar = False)

sorgu = "the football team won the championship match last night"
q_emb = st.encode(sorgu)
skorlar = util.cos_sim(q_emb, embeddings)[0]
en_benzer =skorlar.argsort(descending= True)[:3]

print(f"Sorgu: {sorgu}'\n'")
print("En benzer 3 haber:")
for i in en_benzer:
    i = int(i)
    print(f"   [{ornek_df['category'][i]}] skor = {skorlar[i]:.2f} :: {ornek_df['text'][i][:70]}...")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9922.90it/s]


Sorgu: the football team won the championship match last night'
'
En benzer 3 haber:
   [sport] skor = 0.41 :: Bees handed potential Man Utd tie
 
 Brentford face a home tie against...
   [sport] skor = 0.38 :: Cup holders Man Utd visit Everton
 
 Holders Manchester United and Pre...
   [sport] skor = 0.38 :: Chelsea clinch cup in extra-time
 
 (after extra-time - score at 90 mi...


In [19]:
# 8) Modeli KAYDET -> GERİ YÜKLE -> TAHMİN (production döngüsü)
import pickle 

# (1) KAYDET - tüm pipeline (vectorizer + model) Tek dosyada
with open('bbc_classifier.pkl','wb') as f:
    pickle.dump(model,f)
print("Kaydedildi: bbc_classifier.pkl")

Kaydedildi: bbc_classifier.pkl


# ÖZET

- EDA -> ön işleme -> BoW/TF-IDF -> sınıflandırma -> NER -> semantik arama

- TF-IDF (0.980) > BOW(0.969) final accuracy 0.98 

- NEW varlık çıkarır (ama %100 değil) embeddings anlamca eşleştirir

- Modern alternatif: fine tuned t ransformer (BERT) -- daha doğru, bağlam-duyarlı, ama ağır

- Klasik pipeline: hızlı, ucuz, yorumlanabilir -> güçlü başlangıç